# JumpGuard AI - Dataset Exploration

## Purpose
This notebook explores the Drop Vertical Jump dataset structure before any feature engineering, MediaPipe work, risk modeling, or machine learning.

## What is happening
We safely load `DJ.mat`, identify the MATLAB file format, inspect keys, shapes, object types, nested structures, and load `IK_column_labels.xlsx`.

## Why
The project specification requires understanding the dataset dynamically. No variable names, participant layouts, array dimensions, or label mappings should be invented.

## Expected output
Printed file metadata, structure summaries, workbook summaries, and candidate mappings between IK labels and numeric arrays in `DJ.mat`. If the mapping is ambiguous, execution stops and asks for clarification.

## 1. Setup

### What is happening
Import project utilities and define the dataset paths used in this exploration.

### Why
Keeping paths and imports explicit makes the notebook repeatable from the project root.

### Expected output
No output unless an import or path setup fails.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.inspect_dataset import (
    investigate_label_mapping,
    print_structure_summary,
    summarize_label_workbook,
    summarize_structure,
)
from src.load_dataset import detect_matlab_version, load_excel_workbook, load_mat_dataset
from src.utils import format_bytes, require_file
from src.visualize import plot_sheet_completeness

MAT_PATH = PROJECT_ROOT / "data" / "sample" / "DJ.mat"
IK_LABELS_PATH = PROJECT_ROOT / "data" / "metadata" / "IK_column_labels.xlsx"

## 2. Validate Required Files

### What is happening
Check that `DJ.mat` and `IK_column_labels.xlsx` exist before loading them.

### Why
Clear file validation prevents silent failures and makes missing dataset pieces obvious.

### Expected output
Absolute file paths and readable file sizes.

In [ ]:
for path in (MAT_PATH, IK_LABELS_PATH):
    resolved = require_file(path)
    print(f"Found: {resolved}")
    print(f"Size:  {format_bytes(resolved.stat().st_size)}")

## 3. Detect MATLAB Version

### What is happening
Read the MAT-file header and determine whether the file should be loaded with `scipy.io.loadmat` or `h5py`.

### Why
MATLAB v7.3 files are HDF5-backed and require different loading logic than classic MAT-files.

### Expected output
The detected MATLAB file variant, selected loader, file size, and header text.

In [ ]:
mat_info = detect_matlab_version(MAT_PATH)
print(f"Path:    {mat_info.path}")
print(f"Version: {mat_info.version}")
print(f"Loader:  {mat_info.loader}")
print(f"Size:    {format_bytes(mat_info.size_bytes)}")
print(f"Header:  {mat_info.header}")

## 4. Load DJ.mat Safely

### What is happening
Load the MATLAB dataset using the loader selected by version detection.

### Why
The project must inspect the actual dataset structure before later engineering work.

### Expected output
Top-level keys from the loaded MATLAB object. If a required dependency is missing, the error should clearly name it.

In [ ]:
mat_info, dj_data = load_mat_dataset(MAT_PATH)
print(f"Loaded with: {mat_info.loader}")
print("Top-level keys:")
for key in dj_data.keys():
    print(f"- {key}")

## 5. Inspect Keys, Shapes, Types, and Nested Structures

### What is happening
Recursively summarize dictionaries, MATLAB structs, object arrays, and numeric arrays.

### Why
This reveals the dataset organization without assuming participant layouts, trial structure, or matrix dimensions.

### Expected output
A compact structure table with path, Python type, shape, dtype, and number of dimensions.

In [ ]:
structure_summaries = summarize_structure(dj_data, name="DJ", max_depth=10)
print_structure_summary(structure_summaries)

## 6. Load IK Column Labels

### What is happening
Load every sheet from `IK_column_labels.xlsx`.

### Why
The IK labels are needed to understand which joint-angle columns may correspond to arrays in `DJ.mat`.

### Expected output
Workbook sheet names and a per-sheet summary of rows, columns, column names, and non-null cell counts.

In [ ]:
ik_workbook = load_excel_workbook(IK_LABELS_PATH)
print("Sheets:")
for sheet_name in ik_workbook.keys():
    print(f"- {sheet_name}")

label_summary = summarize_label_workbook(ik_workbook)
label_summary

## 7. Preview Label Sheets

### What is happening
Display the first rows of each IK label sheet.

### Why
A human-readable preview helps determine whether labels are in rows, columns, headers, or a more complex layout.

### Expected output
One preview table per sheet.

In [ ]:
for sheet_name, frame in ik_workbook.items():
    print(f"\nSheet: {sheet_name} | shape={frame.shape}")
    display(frame.head(10))

## 8. Identify the Authoritative IK Label Range

### What is happening
Programmatically inspect rows with 44 populated cells and select the confirmed label row.

### Why
The workbook contains metadata text as well as labels. The label range must be identified from workbook structure, not from a broad string scrape.

### Expected output
The numeric index row `A11:AR11`, the label row `A12:AR12`, and the 44 labels used for `Joint_Angles`.


In [ ]:
ik_sheet_name = "CMJ_dom_t1_IK"
ik_frame = ik_workbook[ik_sheet_name]

# pandas has already promoted row 1 to column headers, so Excel row 11 is iloc[9]
# and Excel row 12 is iloc[10].
ik_index_values = ik_frame.iloc[9, :44].tolist()
ik_label_values = ik_frame.iloc[10, :44].tolist()

print("Index range: CMJ_dom_t1_IK!A11:AR11")
print(ik_index_values)
print("Label range: CMJ_dom_t1_IK!A12:AR12")
print(ik_label_values)

if ik_index_values != list(range(1, 45)):
    raise RuntimeError("Expected Excel row 11 to contain numeric indices 1..44.")

if len(ik_label_values) != 44 or any(not isinstance(label, str) for label in ik_label_values):
    raise RuntimeError("Expected Excel row 12 to contain 44 text IK labels.")

print(f"Confirmed IK label count: {len(ik_label_values)}")


## 9. Investigate Label-to-MAT Mapping

### What is happening
Search numeric arrays in `DJ.mat` for dimensions matching the confirmed 44 IK labels.

### Why
A 44-column numeric array is the expected target for the IK label mapping, but we still inspect candidates rather than assuming a variable name.

### Expected output
Candidate array paths and matching axes. Review candidates before using the mapping downstream.


In [ ]:
mapping_candidates = investigate_label_mapping(structure_summaries, len(ik_label_values))
display(mapping_candidates)

if mapping_candidates.empty:
    raise RuntimeError(
        "No numeric array dimension matched the confirmed IK label count. "
        "Stop here and inspect the MATLAB nesting and label workbook again."
    )

print("Candidate mappings found. `Joint_Angles` records should be reviewed as the authoritative 44-column arrays.")


## 10. Visualize Label Workbook Completeness

### What is happening
Plot non-null cell counts for each sheet in the IK label workbook.

### Why
This gives a quick sense of where label information is concentrated.

### Expected output
A simple bar chart. This is exploratory context only, not a feature engineering step.

In [ ]:
plot_sheet_completeness(ik_workbook)

## Exploration Notes

Use this section to record confirmed findings only after reviewing the outputs above.

- Confirmed MATLAB file version:
- Confirmed top-level dataset keys:
- Confirmed IK label source cells or columns:
- Confirmed mapping between IK labels and `DJ.mat`:
- Open questions: